# Meta-train a projection with a ridge head (design D5)

Trains the projection of `transfer_models.MetaRidgeEncoder` so that a ridge regression fitted on K frames of a robot predicts the joints of that robot well (see `DESIGN-Transfer.md`, D5). The backbone (VGG19, pooled) is frozen, so its features are computed once.

Each episode does the following:
1. It takes a random robot of `exp["robots"]`.
2. It takes K support frames from one demonstration, and query frames from the other demonstrations.
3. With probability `joint_subset_prob`, it keeps a random subset of the joints.
4. The loss is that of the ridge regression fitted on the support set, measured on the query set (`inner_solver: ridge`). Alternatively, the head and the projection are adapted with a few SGD steps on the support set, with first-order MAML (`inner_solver: fomaml`).

The model with the best loss on fixed validation episodes is kept.

In [ ]:
import sys
sys.path.append("../..")

from exp_run_config import Config
Config.PROJECTNAME = "BerryPicker"

import copy
import math
import pathlib
import pandas as pd
import torch
import torch.nn.functional as F

from exprunflow.reproducibility import configure_deterministic_run
from training_harness.checkpoints import model_file
from visual_proprioception.transfer.transfer_data import load_frames, to_float
from visual_proprioception.transfer.transfer_models import (
    MetaRidgeEncoder, fomaml_task, ridge_fit, ridge_predict)

In [ ]:
# *** Initialize the variables with default values
# *** This cell should be tagged as parameters
# *** If papermill is used, some of the values will be overwritten

# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"

experiment = "sensorprocessing_transfer_meta_ridge"
run = "meta-ridge_128_hand+panda"
# If not None, set the number of meta steps to something different than the exp
meta_steps = None

# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

In [ ]:
if expruns_path:
    Config().set_exprun_path(pathlib.Path(expruns_path))
if results_path:
    Config().set_results_path(pathlib.Path(results_path))

exp = Config().get_experiment(experiment, run, creation_style=creation_style)
seed = configure_deterministic_run(exp["seed"])
device = Config().runtime["device"]
model = MetaRidgeEncoder(exp)
modelfile = model_file(exp)
will_load_existing = modelfile.exists() and exp.get("reload_existing_model", True)

In [ ]:
# the frozen features of every frame of every robot, computed once
data = {}
if not will_load_existing:
    for robot in exp["robots"]:
        exp_robot = Config().get_experiment(robot["robot_exp"], robot["robot_run"])
        for split in ["training_data", "validation_data"]:
            frames = load_frames(robot[split], exp["image_size"], exp_robot, exp["frame_step"])
            n = frames["images"].shape[0]
            features = torch.cat([model.features(to_float(frames["images"][i:i + 64]).to(device))
                                  for i in range(0, n, 64)])
            data[(robot["name"], split)] = {"features": features, "targets": frames["targets"].to(device),
                                            "demo_index": frames["demo_index"]}
            print(f'{robot["name"]} {split}: {n} frames, features {tuple(features.shape)}')

In [ ]:
def sample_task(split, generator):
    """One task: a robot, K support frames from one demonstration, query frames
    from the other demonstrations, and possibly a random subset of the joints"""
    robot = exp["robots"][torch.randint(len(exp["robots"]), (1,), generator=generator).item()]
    d = data[(robot["name"], split)]
    demos = d["demo_index"].unique()
    support_demo = demos[torch.randint(len(demos), (1,), generator=generator).item()]
    support_rows = (d["demo_index"] == support_demo).nonzero().squeeze(1)
    query_rows = (d["demo_index"] != support_demo).nonzero().squeeze(1)
    assert len(query_rows) > 0, f"{robot['name']} {split} needs at least two demonstrations"
    k = exp["support_sizes"][torch.randint(len(exp["support_sizes"]), (1,), generator=generator).item()]
    support_rows = support_rows[torch.randperm(len(support_rows), generator=generator)[:k]].to(device)
    query_rows = query_rows[torch.randperm(len(query_rows), generator=generator)[:exp["query_size"]]].to(device)
    joints = torch.arange(d["targets"].shape[1])
    if len(joints) > 2 and torch.rand(1, generator=generator).item() < exp["joint_subset_prob"]:
        count = torch.randint(2, len(joints) + 1, (1,), generator=generator).item()
        joints = joints[torch.randperm(len(joints), generator=generator)[:count]]
    joints = joints.to(device)
    return (d["features"][support_rows], d["targets"][support_rows][:, joints],
            d["features"][query_rows], d["targets"][query_rows][:, joints])


def ridge_task_loss(task):
    f_s, y_s, f_q, y_q = task
    w = ridge_fit(model.projection(f_s), y_s, model.log_lambda)
    return F.mse_loss(ridge_predict(model.projection(f_q), w), y_q)


def task_loss(task):
    """The query loss of a task, without gradients"""
    if exp["inner_solver"] == "ridge":
        with torch.no_grad():
            return ridge_task_loss(task).item()
    # FOMAML needs the gradients of its inner loop
    return fomaml_task(model.projection, *task, exp["inner_steps"], exp["inner_lr"])[0].item()

In [ ]:
if will_load_existing:
    model.load_state_dict(torch.load(modelfile, map_location=device, weights_only=True))
else:
    steps = meta_steps if meta_steps is not None else exp["meta_steps"]
    validation_tasks = [sample_task("validation_data", torch.Generator().manual_seed(1000 + i))
                        for i in range(exp["validation_tasks"])]
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=exp["learning_rate"])
    generator = torch.Generator().manual_seed(seed)
    best_loss, best_state, history = math.inf, None, []
    for step in range(1, steps + 1):
        model.train()
        optimizer.zero_grad()
        tasks = [sample_task("training_data", generator) for _ in range(exp["tasks_per_step"])]
        if exp["inner_solver"] == "ridge":
            loss = sum(ridge_task_loss(task) for task in tasks) / len(tasks)
            loss.backward()
            loss = loss.item()
        else:
            loss = 0
            for task in tasks:
                query_loss, grads = fomaml_task(model.projection, *task, exp["inner_steps"], exp["inner_lr"])
                for name, p in model.projection.named_parameters():
                    g = grads[name] / len(tasks)
                    p.grad = g if p.grad is None else p.grad + g
                loss += query_loss.item() / len(tasks)
        optimizer.step()
        if step % exp["validation_interval"] == 0 or step == steps:
            model.eval()
            validation_loss = sum(task_loss(task) for task in validation_tasks) / len(validation_tasks)
            history.append({"step": step, "training_loss": loss, "validation_loss": validation_loss,
                            "lambda": model.log_lambda.exp().item()})
            print(history[-1])
            if validation_loss < best_loss:
                best_loss, best_state = validation_loss, copy.deepcopy(model.state_dict())
    torch.save(best_state, modelfile)
    pd.DataFrame(history).to_csv(pathlib.Path(exp["data_dir"], "meta_training.csv"), index=False)
    print(f"Saved the best model (validation loss {best_loss:.5f}) to {modelfile}")

In [ ]:
exp.done()